# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking, built on a binary classifier.** My lane (freestyle: growth / recovery / momentum) asks "which pages should a reviewer look at first?" That is a queue, so what I really need is a good *order*, not a yes/no answer for every page. I train a classifier on an observed yes/no outcome and then sort pages by its score.

Why not the other task types:
- *Clustering* would describe kinds of pages, but nobody can act on "cluster 3" without a target.
- *Regression* on next month's impressions would mostly learn page size (big pages stay big). The interesting part is the change, and a reviewer only needs to know whether the change is bad enough to look at.
- A plain *score* with hand-set weights is exactly what my baseline is (w04). The model has to beat it.

In [ ]:
# The task in one place: a risk score per (page, decision date), sorted into a review queue.
task = {'type': 'binary classification, used as a RANKING (score -> per-site review queue)',
        'unit': 'one content item (page) at one decision date T',
        'label': 'declined = page fell >20% further than its own site over (T, T+30], vs its prior 60-day monthly average',
        'features_window': '(T-90, T]  -- strictly before the label month',
        'primary metric': 'precision@50 next to the base rate; per-site precision@10; ROC-AUC / average precision for the whole ranking'}
for k, v in task.items():
    print(f'{k:16} {v}')

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: did the page fall behind its own site next month?**

For every page and decision date T (end of a month):
- `baseline_monthly_imp` = the page's average 30-day impressions over the 60 days before T
- `page_ratio` = impressions in the 30 days after T ÷ that baseline
- `client_ratio` = the same ratio for the whole site (sum over the site's pages)
- **declined = 1 if page_ratio < 0.8 × client_ratio**, i.e. the page did more than 20% worse than its own site.

This is an **observed outcome** from a later window. It is not a rule applied to the same days the features come from (that was the weakness of the starter's `trend_direction` label).

I started with the plain version ("impressions fell more than 20%"). The table below shows why I dropped it: across the three months its rate climbs from 52% to 64% to 75% because whole sites were losing impressions at the same time (the median site kept only 89%, then 74%, then 64% of its traffic). A refresh can't fix a site-wide drop caused by an algorithm update, a tracking change or a season. Measuring each page against its own site removes that, and the rate stays flat at about 49%, 49%, 52%.

In [ ]:
# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')

# The label is an OBSERVED future outcome: what happened in the 30 days after T.
show = ['decision_date', 'imp_prev30', 'imp_last30', 'baseline_monthly_imp', 'imp_next30', 'page_ratio', 'client_ratio', 'declined', 'declined_abs']
print(m[show].sample(6, random_state=ff.SEED).round(3).to_string(index=False))
print()
# Why site-relative? The plain ">20% drop" label drifts with whole-portfolio swings.
print(m.groupby('decision_date').agg(rows=('declined', 'size'), clients=('client_hash_id', 'nunique'),
      site_relative_rate=('declined', 'mean'), plain_drop_rate=('declined_abs', 'mean'),
      median_site_ratio=('client_ratio', 'median')).round(3).to_string())

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50, always printed next to the base rate.** A reviewer works down a list, so the honest question is "of the first 50 pages I open, how many really were falling behind?". Because the label is site-relative, I also report **per-site precision@10**: inside each site, how many of that site's top-10 flagged pages were right. That is closer to how a single site would actually use the queue.

What counts as good: random picking gets about the base rate (≈0.49, see below). The baseline rule has to clear that clearly, and the model has to beat the rule on the same rows. I also track ROC-AUC and average precision because they judge the whole ordering, not just the top.

In [ ]:
# What would a reviewer get by picking pages at random? The base rate - that is the floor.
for T, g in m.groupby('decision_date'):
    y = g.declined.astype(int).values
    rnd = [ff.precision_at_k(np.random.default_rng(s).random(len(g)), y, 50) for s in range(200)]
    print(f'{T}: base rate {y.mean():.3f} | random-queue precision@50 = {np.mean(rnd):.3f} '
          f'(5th-95th pct {np.percentile(rnd, 5):.2f}-{np.percentile(rnd, 95):.2f})')

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page at one decision date.** Two development dates: 2026-03-31 (outcome month = April) and 2026-04-30 (outcome month = May). June 2026 is kept sealed for the final test in w06.

The model population keeps pages that:
- had at least 200 impressions in the 60 days before T (below that a "drop" is mostly noise),
- were at least 90 days old at T, so the whole feature window exists,
- belong to a client whose Search Console feed covers the full 90 days (otherwise "no data yet" would look like "no traffic").

The grain check below confirms there is exactly one row per page per date.

In [ ]:
# One row = one content item at one decision date. Check the grain and show the frame.
dupes = panel.groupby(['decision_date', 'content_hash_id']).size()
print('max rows per (decision_date, content):', int(dupes.max()), '-> grain holds' if dupes.max() == 1 else '-> GRAIN BROKEN')
print(f"{m.content_hash_id.nunique():,} distinct pages, {m.client_hash_id.nunique()} clients in the model population")
print(m[['decision_date', 'client_hash_id', 'content_hash_id'] + ff.FEATURES[:7] + ['declined']].head(6).round(2).to_string(index=False))

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because the signals interact, and some point the opposite way from what people expect.**

The cross-tab below splits pages three ways: momentum, whether last month was carried by a single spike day, and age. The decline rate moves a lot between cells, and not in a way one threshold captures. Falling pages are riskier, but a smooth falling page and a spiky falling page are different cases. Older pages are actually *less* likely to fall behind their site than 3-6 month old ones, which is the opposite of the usual "old content decays" rule. On top of that the model also gets position, its drift and volatility, CTR, word count and keyword volume.

A hand-written if-statement can encode two or three of these. It cannot weigh eighteen of them against each other. That is the gap a model can fill, and w05 checks whether it actually does better than the rule on clients it has never seen.

In [ ]:
# Is one if-statement enough? Cross three obvious signals and read the decline rate in each cell.
mm = m.copy()
mm['momentum'] = pd.cut(mm.imp_momentum, [-20, -0.15, 0.15, 20], labels=['falling', 'flat', 'rising'])
mm['spiky'] = np.where(mm.spike_share > 0.25, 'spiky', 'smooth')
mm['age'] = pd.cut(mm.content_age_days, [90, 180, 365, 10000], labels=['3-6m', '6-12m', '12m+'])
tab = mm.pivot_table(index=['momentum', 'spiky'], columns='age', values='declined', aggfunc='mean', observed=True)
cnt = mm.pivot_table(index=['momentum', 'spiky'], columns='age', values='declined', aggfunc='size', observed=True)
print('decline rate'); print(tab.round(3).to_string()); print('\nrows'); print(cnt.to_string())

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.